In [1]:
import malthusjax as mjx
import jax
import jax.numpy as jnp
import jax.random as jar

# Verify we are running on the intended backend
print(f"MalthusJAX Version: {mjx.__version__}")
print(f"JAX Backend: {jax.devices()[0]}")

# Master Random Key
key = jar.PRNGKey(42)

MalthusJAX Version: 0.2.0
JAX Backend: TFRT_CPU_0


In [2]:
# 1. Define the Problem (100-bit string)
genome_config = mjx.BinaryGenomeConfig(length=100)

# 2. Define Engine Parameters (Static)
# We want 1000 individuals, running for 50 generations
# We preserve the top 5 individuals (Elitism)
params = mjx.StandardEngineParams(
    pop_size=177,
    num_generations=50,
    elitism=5
)

# 3. Initialize Random Population (Batch Creation)
key, k_pop = jar.split(key)
initial_pop = mjx.BinaryPopulation.init_random(k_pop, genome_config, params.pop_size)

print(f"Population Shape: {initial_pop.genes.bits.shape}")

Population Shape: (177, 100)


In [3]:
# 4. Assemble the Engine
# We use the clean 'mjx' namespace we built
engine = mjx.StandardGeneticEngine(
    evaluator=mjx.BinarySumEvaluator(mjx.BinarySumConfig(maximize=True)),
    selection=mjx.selection.Tournament(num_selections=params.pop_size, tournament_size=3),
    crossover=mjx.crossover.Uniform(num_offspring=2, crossover_rate=0.8),
    mutation=mjx.mutation.BitFlip(num_offspring=1, mutation_rate=0.01)
)

# 5. Create Initial State
# This runs one evaluation to populate the 'best_fitness' and 'fitness' fields
key, k_init = jar.split(key)
state = engine.init_state(k_init, initial_pop)

print(f"Initial Best Fitness: {state.best_fitness}")

Initial Best Fitness: 63.0


In [4]:
# 6. Run Evolution (JIT Compiled)
print("Compiling & Running...")

# The .run() method handles the jax.lax.scan loop automatically
final_state, history, elapsed = engine.run(
    state, 
    params, 
    time_it=True, 
    verbose=True
)

# 7. Analyze Results
print(f"\nOptimization Complete in {elapsed:.4f}s")
print(f"Final Best Fitness: {final_state.best_fitness}/{genome_config.length}")

# Check if we solved it (Should be close to 100)
best_genes = final_state.best_genome.genes.bits  # Index into population to get first genome
print(f"Best Genome Sample: {best_genes}")

Compiling & Running...
Running evolution (JIT compilation automatic)

Optimization Complete in 0.4074s
Final Best Fitness: 100.0/100
Best Genome Sample: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]

Optimization Complete in 0.4074s
Final Best Fitness: 100.0/100
Best Genome Sample: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


# Part 2: Knapsack Problem
Solving the 0/1 Knapsack problem: selecting items with given weights and values to maximize total value without exceeding capacity.

In [5]:
# 1. Setup Knapsack Problem Data
num_items = 50
key, k_data = jar.split(key)

# Generate random weights and values
weights = jar.randint(k_data, (num_items,), 1, 20).astype(jnp.float32)
values = jar.randint(k_data, (num_items,), 10, 100).astype(jnp.float32)
capacity = jnp.sum(weights) * 0.6  # Capacity is 60% of total weight

print(f"Knapsack Capacity: {capacity}")

# 2. Configure Genome and Evaluator
knapsack_genome_config = mjx.BinaryGenomeConfig(length=num_items)
knapsack_config = mjx.KnapsackConfig(
    weights=weights, 
    values=values, 
    capacity=capacity,
)
knapsack_evaluator = mjx.KnapsackEvaluator(knapsack_config)

# 3. Initialize Population
key, k_pop = jar.split(key)
knapsack_pop = mjx.BinaryPopulation.init_random(k_pop, knapsack_genome_config, params.pop_size)

# 4. Create Engine
knapsack_engine = mjx.StandardGeneticEngine(
    evaluator=knapsack_evaluator,
    selection=mjx.selection.Tournament(num_selections=params.pop_size, tournament_size=3),
    crossover=mjx.crossover.Uniform(num_offspring=2, crossover_rate=0.8),
    mutation=mjx.mutation.BitFlip(num_offspring=1, mutation_rate=0.02)
)

# 5. Run Evolution
print("\nRunning Knapsack Evolution...")
key, k_init = jar.split(key)
ks_state = knapsack_engine.init_state(k_init, knapsack_pop)

final_ks_state, ks_history, ks_elapsed = knapsack_engine.run(
    ks_state, 
    params, 
    time_it=True, 
    verbose=True
)

print(f"Knapsack Solved in {ks_elapsed:.4f}s")
print(f"Best Value: {final_ks_state.best_fitness}")

Knapsack Capacity: 356.4000244140625

Running Knapsack Evolution...

Running Knapsack Evolution...
Running evolution (JIT compilation automatic)
Running evolution (JIT compilation automatic)
Knapsack Solved in 0.3935s
Best Value: 2519.0
Knapsack Solved in 0.3935s
Best Value: 2519.0


# Part 3: Real-Valued Optimization (Sphere Function)
Optimizing a continuous function (Sphere) using Real Genomes.
Objective: Minimize f(x) = sum(x^2). Target is 0.

In [6]:
# 1. Setup Real Genome Problem
dim = 10
real_config = mjx.RealGenomeConfig(
    length=dim, 
    bounds=(-5.12, 5.12)
)

# 2. Configure Evaluator (Minimize Sphere)
sphere_evaluator = mjx.SphereEvaluator(mjx.SphereConfig(maximize=True))

# 3. Initialize Population
key, k_pop = jar.split(key)
real_pop = mjx.RealPopulation.init_random(k_pop, real_config, params.pop_size)

# 4. Create Engine with Real Operators
real_engine = mjx.StandardGeneticEngine(
    evaluator=sphere_evaluator,
    selection=mjx.selection.Tournament(num_selections=params.pop_size, tournament_size=3),
    # Use Blend Crossover for Real Genomes
    crossover=mjx.crossover.Blend(num_offspring=3, alpha=0.5),
    # Use Gaussian Mutation
    mutation=mjx.mutation.Gaussian(num_offspring=2, mutation_rate=0.2, mutation_strength=0.5)
)

# 5. Run Evolution
print("\nRunning Sphere Evolution...")
key, k_init = jar.split(key)
real_state = real_engine.init_state(k_init, real_pop)
print(f"Initial Best Fitness: {real_state.best_fitness}")

final_real_state, real_history, real_elapsed = real_engine.run(
    real_state, 
    params, 
    time_it=True, 
    verbose=True
)

print(f"Sphere Optimization in {real_elapsed:.4f}s")
print(f"Best Fitness (Closer to 0 is better): {final_real_state.best_fitness}")
print(f"Best Solution: {final_real_state.best_genome}")


Running Sphere Evolution...
Initial Best Fitness: 148.09080505371094
Running evolution (JIT compilation automatic)
Sphere Optimization in 0.5367s
Best Fitness (Closer to 0 is better): 262.14398193359375
Best Solution: RealPopulation(genes=<RealGenome([-5.120, -5.120, 5.120, ..., 5.120], len=10)>, fitness=Array(262.14398, dtype=float32), config=RealGenomeConfig(length=10, bounds=(-5.12, 5.12)))
Sphere Optimization in 0.5367s
Best Fitness (Closer to 0 is better): 262.14398193359375
Best Solution: RealPopulation(genes=<RealGenome([-5.120, -5.120, 5.120, ..., 5.120], len=10)>, fitness=Array(262.14398, dtype=float32), config=RealGenomeConfig(length=10, bounds=(-5.12, 5.12)))


In [7]:
#pyfrom malthusjax.core.fitness.real_evaluators import MSEConfig, MSEEvaluator

# Part 4: Linear Regression with Genetic Algorithm
# Using Linear Genomes to find optimal weights and bias for a regression problem

# 1. Generate synthetic regression data
key, k_data = jar.split(key)
n_samples = 100
n_features = 5

# True weights and bias (what we're trying to find)
true_weights = jnp.array([2.5, -1.3, 0.8, 3.2, -0.5])
true_bias = 1.5

# Generate random features
X = jar.normal(k_data, (n_samples, n_features))
# Generate target with some noise
key, k_noise = jar.split(key)
y = X @ true_weights + true_bias + jar.normal(k_noise, (n_samples,)) * 0.1

print(f"Data shape: X={X.shape}, y={y.shape}")
print(f"True weights: {true_weights}")
print(f"True bias: {true_bias}")

regression_data = (X,y)

# 2. Configure Linear Genome (n_features weights + 1 bias)
linear_config = mjx.LinearGenomeConfig(
    length= 73,
    num_inputs=n_features,
    num_ops = 29,
    max_arity=3,
    X= X,
    y= y,
    maximize =  False

)



# 4. Initialize Population
key, k_pop = jar.split(key)
linear_pop = mjx.LinearPopulation.init_random(k_pop,
                                              linear_config,
                                              params.pop_size)


# 5. Create Engine with Real-valued operators
regression_engine = mjx.StandardGeneticEngine(
    evaluator=mjx.LinearGPEvaluator(config=linear_config),
    selection=mjx.selection.Tournament(num_selections=params.pop_size, tournament_size=5),
    crossover=mjx.crossover.Linear(num_offspring=3),
    mutation=mjx.mutation.Linear(num_offspring=2, op_rate=0.3, arg_rate=0.5)
)

# 6. Run Evolution
print("\nRunning Linear Regression Evolution...")
key, k_init = jar.split(key)


Data shape: X=(100, 5), y=(100,)
True weights: [ 2.5 -1.3  0.8  3.2 -0.5]
True bias: 1.5

Running Linear Regression Evolution...

Running Linear Regression Evolution...


In [8]:
reg_state = regression_engine.init_state(k_init, linear_pop)


evaluator=mjx.LinearGPEvaluator(config=linear_config)

fitness = evaluator.evaluate_batch(reg_state.population)


# split keys for two tournaments
key, k1, k2 = jar.split(key, 3)

pop_1_index = mjx.operators.TournamentSelection(num_selections=10, tournament_size=5)(key = k1, fitness=fitness)
pop_2_index = mjx.operators.TournamentSelection(num_selections=10, tournament_size=5)(key = k2, fitness=fitness)

pop1 = reg_state.population[pop_1_index]
pop2 = reg_state.population[pop_2_index]

In [9]:
reg_state = regression_engine.init_state(k_init, linear_pop)
print(f"Initial Best Fitness (negative MSE): {reg_state.best_fitness}")

# Use more generations for regression
reg_params = mjx.StandardEngineParams(
    pop_size=params.pop_size,
    num_generations=100,
    elitism=10
)

final_reg_state, reg_history, reg_elapsed = regression_engine.run(
    reg_state, 
    reg_params, 
    time_it=True, 
    verbose=True
)

# 7. Analyze Results
print(f"\nRegression Optimization Complete in {reg_elapsed:.4f}s")
print(f"Final Best Fitness (negative MSE): {final_reg_state.best_fitness}")

# Extract learned parameters


Initial Best Fitness (negative MSE): -10.500398635864258
Running evolution (JIT compilation automatic)

Regression Optimization Complete in 1.2781s

Regression Optimization Complete in 1.2781s
Final Best Fitness (negative MSE): -18.774858474731445
Final Best Fitness (negative MSE): -18.774858474731445


In [10]:
best_genome = final_reg_state.best_genome
best_genome.genes

<LinearGenome(L=(73,))>